# A new (fictional) Indian female voice → RVC v2 model — Colab T4

No real person's voice is used: VoxCPM2 (open source) invents a voice, makes ~10 minutes of speech in it, and RVC v2
(Applio, MIT) learns that voice. The RVC model then turns any speech (Gemini / Piper / browser, Hindi or English)
into this voice for the website avatar.

**VS Code:** Select Kernel → Colab → New Colab Server → GPU (T4). **colab.google:** Runtime → Change runtime type → T4 GPU.
Run the cells in order. If a cell fails, the cause is inside the red error message — copy that text.
Total ≈ 45–70 min (data ~10 min, RVC install ~10 min, training ~20–40 min).

In [4]:
#@title 1) Install VoxCPM (if pip asks to restart: Runtime → Restart session, then run cell 0 and 2)
!pip install -q -U voxcpm soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.1 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 26.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 12.1 MB/s eta 0:00:00
  Installing build dep

In [5]:
#@title 2) Load VoxCPM2 and make 6 new voices — listen and pick the best young Indian female voice
import inspect, numpy as np, torch, torch._dynamo, soundfile as sf
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True
from voxcpm import VoxCPM
import IPython.display as ipd

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=True, optimize=False)
SR = model.tts_model.sample_rate
SIG = inspect.signature(model.generate).parameters
print("generate() parameters:", list(SIG))

def say(text, ref=None, ref_text=None, steps=10, seed=None):
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    kw = {"text": text}
    if "inference_timesteps" in SIG: kw["inference_timesteps"] = steps
    if ref:
        k = next((k for k in ("prompt_wav_path", "reference_wav_path", "prompt_audio_path") if k in SIG), None)
        if not k: raise RuntimeError(f"no reference-audio parameter in {list(SIG)}")
        kw[k] = ref
        kt = next((k for k in ("prompt_text", "reference_text") if k in SIG), None)
        if kt: kw[kt] = ref_text
    out = model.generate(**kw)
    parts = [out] if isinstance(out, (np.ndarray, torch.Tensor)) else list(out)
    return np.concatenate([np.asarray(p.detach().cpu() if hasattr(p, "detach") else p, dtype=np.float32).reshape(-1) for p in parts])

GREETING = "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
os.makedirs("/content/cand", exist_ok=True)
for seed in range(6):
    w = say(GREETING, seed=seed)
    sf.write(f"/content/cand/voice_{seed}.wav", w, SR)
    print(f"--- voice {seed}"); ipd.display(ipd.Audio(w, rate=SR))
print("None good? change range(6) to range(6, 12) and run again.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

voxcpm_model_path: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/32279effe8c19989596f05d353d1447f51d9e915, zipenhancer_model_path: iic/speech_zipenhancer_ans_multiloss_16k_base, enable_denoiser: True
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/32279effe8c19989596f05d353d1447f51d9e915/audiovae.pth
Running on device: cpu, dtype: bfloat16
Loading model from safetensors: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/32279effe8c19989596f05d353d1447f51d9e915/model.safetensors
Loaded VoxCPM2Model


configuration.json: 0.00B [00:00, ?B/s]

2026-10-12 02:30:20,852 | INFO    | modelscope_hub.download | Downloading 11 files from iic/speech_zipenhancer_ans_multiloss_16k_base@master


... (22 outputs left out)

--- voice 4


 33%|███▎      | 41/124 [02:24<04:52,  3.53s/it]


--- voice 5


None good? change range(6) to range(6, 12) and run again.


In [ ]:
View Jupyter log for further details. Unable to get resolved server information for google.colab:colab:r-5b8bb495-4f61-4f38-b5ac-b00be326677c fix
    

In [ ]:
#@title 3) ~10 minutes of speech in the chosen voice (English, Hinglish, Hindi — showroom talk)
CHOSEN = 0   #@param {type:"integer"}
REF, REF_TEXT = f"/content/cand/voice_{CHOSEN}.wav", GREETING
models = ["Fortuner", "Innova Hycross", "Innova Crysta", "Glanza", "Urban Cruiser Hyryder", "Camry", "Legender", "Hilux",
          "Rumion", "Taisor", "Creta", "Venue", "Verna", "Alcazar", "Exter", "i20"]
en = ["The {m} is one of our most popular cars this month.", "Would you like to book a test drive for the {m}?",
      "The {m} comes with six airbags and a five star safety rating.", "I can show you the {m} page on our website right now.",
      "Our showroom in Moti Nagar is open from ten in the morning till seven in the evening.",
      "The on-road price depends on the variant and the insurance you choose.", "Can I have your name and phone number, please?",
      "Your test drive is confirmed for tomorrow at eleven thirty.", "Thank you for calling, have a wonderful day!",
      "The waiting period for the {m} is about four to six weeks right now.", "We also have easy EMI options starting from a low down payment.",
      "Is there anything else I can help you with today?"]
hinglish = ["{m} ke baare mein main aapko poori jaankari de sakti hoon.", "Kya aap {m} ki test drive book karna chahenge?",
            "Aap kal subah das baje showroom aa sakte hain.", "{m} mein petrol aur hybrid dono options milte hain.",
            "Aapka naam aur mobile number bata dijiye, please.", "Bilkul, main abhi aapke liye slot check karti hoon.",
            "Is mahine {m} par achha offer chal raha hai.", "Koi aur sawaal ho toh zaroor poochiye."]
hindi = ["नमस्ते! गैलेक्सी टोयोटा में आपका स्वागत है।", "आपकी टेस्ट ड्राइव कल सुबह ग्यारह बजे के लिए पक्की हो गई है।",
         "हमारा शोरूम मोती नगर में है और रविवार को भी खुला रहता है।", "क्या मैं आपकी और कोई मदद कर सकती हूँ?",
         "इस गाड़ी में सात एयरबैग और बड़ी डिक्की मिलती है।", "धन्यवाद! आपका दिन शुभ हो।"]
lines = []
for i, m in enumerate(models):
    lines += [en[i % len(en)].format(m=m), en[(i + 5) % len(en)].format(m=m), hinglish[i % len(hinglish)].format(m=m),
              hinglish[(i + 3) % len(hinglish)].format(m=m), hindi[i % len(hindi)]]
lines += [e.format(m="car") for e in en] + [h.format(m="ye gaadi") for h in hinglish]
os.makedirs("/content/dataset", exist_ok=True)
total, t0 = 0.0, time.time()
for i, text in enumerate(lines):
    w = say(text, ref=REF, ref_text=REF_TEXT)
    sf.write(f"/content/dataset/{i:03d}.wav", w, SR)
    total += len(w) / SR
    if i % 10 == 0: print(f"{i + 1}/{len(lines)}  {total / 60:.1f} min  ({time.time() - t0:.0f} s)")
print(f"dataset: {len(lines)} clips, {total / 60:.1f} minutes")
ipd.display(ipd.Audio("/content/dataset/002.wav"))
# a test sentence in a DIFFERENT voice: RVC must turn it into the chosen voice in cell 7
sf.write("/content/test_other_voice.wav", say("Namaste! Main Galaxy Toyota se baat kar rahi hoon. Fortuner ki test drive kab rakhein?", seed=CHOSEN + 50), SR)
del model; import gc; gc.collect(); torch.cuda.empty_cache()   # free the GPU for RVC

In [ ]:
#@title 4) RVC v2 (Applio) in its own Python 3.10 environment (~10 min)
run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba")
if not os.path.exists("/root/.local/share/mamba/envs/rvc") and not os.path.exists("/root/micromamba/envs/rvc"):
    run("micromamba create -y -q -n rvc -c conda-forge python=3.10")
if not os.path.exists("/content/Applio"):
    run("git clone -q --depth 1 https://github.com/IAHispano/Applio.git /content/Applio")
run("micromamba run -n rvc pip install -q -r requirements.txt", cwd="/content/Applio", tail=40)
print(run("micromamba run -n rvc python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\""))
# pretrained RVC v2 base models, RMVPE pitch model, ContentVec embedder
print(run("micromamba run -n rvc python core.py prerequisites --pretraineds_hifigan True --models True --exe False",
          cwd="/content/Applio", tail=40)[-1500:])

In [ ]:
#@title 5) Train the voice (preprocess → features → training → index)
NAME = "galaxy_voice"
EPOCHS = 150   #@param {type:"integer"}
A = "/content/Applio"
def core(*args, tail=40):
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
t0 = time.time()
print(core("preprocess", "--model_name", NAME, "--dataset_path", "/content/dataset", "--sample_rate", 40000,
           "--cpu_cores", os.cpu_count())[-800:])
print(core("extract", "--model_name", NAME, "--f0_method", "rmvpe", "--sample_rate", 40000, "--gpu", 0,
           "--cpu_cores", os.cpu_count())[-800:])
print(core("train", "--model_name", NAME, "--total_epoch", EPOCHS, "--save_every_epoch", 25, "--sample_rate", 40000,
           "--batch_size", 8, "--gpu", 0, "--pretrained", True, "--save_only_latest", True, tail=60)[-1500:])
print(core("index", "--model_name", NAME)[-800:])
print(f"done in {(time.time() - t0) / 60:.0f} min")
print("\n".join(sorted(glob.glob(f"{A}/logs/{NAME}/*.pth") + glob.glob(f"{A}/logs/{NAME}/*.index"))))

In [ ]:
#@title 6) Test: a different voice → our new voice
pth = sorted(glob.glob(f"{A}/logs/{NAME}/{NAME}*.pth"), key=os.path.getmtime)[-1]
idx = sorted(glob.glob(f"{A}/logs/{NAME}/*.index"), key=os.path.getmtime)[-1]
print(pt
      h, idx)
print(core("infer", "--input_path", "/content/test_other_voice.wav", "--output_path", "/content/test_converted.wav",
           "--pth_path", pth, "--index_path", idx, "--pitch", 0, "--f0_method", "rmvpe", "--index_rate", 0.75)[-600:])
print("before (other voice):"); ipd.display(ipd.Audio("/content/test_other_voice.wav"))
print("after (our voice):");    ipd.display(ipd.Audio("/content/test_converted.wav"))
print("reference (chosen voice):"); ipd.display(ipd.Audio(REF))

In [ ]:
#@title 7) Download the model (put the zip in voice_clone/work/rvc/ on the laptop)
import shutil
os.makedirs("/content/out", exist_ok=True)
for f in (pth, idx, REF):
    shutil.copy(f, "/content/out/")
open("/content/out/README.txt", "w").write("Fictional voice made with VoxCPM2 (seed %d) and RVC v2 (Applio). No real person's voice.\n" % CHOSEN)
shutil.make_archive("/content/galaxy_voice_rvc", "zip", "/content/out")
print(os.path.getsize("/content/galaxy_voice_rvc.zip") // 1_000_000, "MB")
try:
    from google.colab import files
    files.download("/content/galaxy_voice_rvc.zip")
except Exception as ex:
    print("download not available here:", type(ex).__name__, "- use the Colab file browser (/content/galaxy_voice_rvc.zip)")